In [ ]:
# Wei Alexander Xin - wax1
# C4: Convolution using OpenAI Triton

In [ ]:
# Installing needed package (triton)
! pip install triton

In [ ]:
# Importing needed libraries
import torch
import torch.nn.functional as F
import triton
import triton.language as tl

In [ ]:
# Global Variables
DEVICE = "cuda"
C_OUT = 64
C_IN = 3
H = 1024
W = 1024
FH = 3
FW = 3

In [ ]:
# Making torch tensors
tensor_I = torch.rand(1, C_IN, H, W, device=DEVICE)          # Input, assuming that batch_size is one
tensor_F = torch.rand(C_OUT, C_IN, FH, FW , device=DEVICE)   # Weights

In [ ]:
# This is the result from Convolutional Layer provided by Torch
# Use this for correctness check

# WAX Note to grader: F.conv2d computes cross-correlation (no filter flip),
# so our kernel matches that behavior to pass the assert below
golden_out = F.conv2d(tensor_I, tensor_F, padding=1)
print(golden_out.shape) # (1, C_OUT, OUT_H, OUT_W)

In [ ]:
@triton.jit
def my_triton_kernel(
    input_ptr, kernel_ptr, output_ptr,
    
    # Tensor dimensions + padding
    C_IN: tl.constexpr, H: tl.constexpr, W: tl.constexpr,
    C_OUT: tl.constexpr, FH: tl.constexpr, FW: tl.constexpr,
    PAD: tl.constexpr,
):
    """    
    This is a triton kernel that does Conv assuming that padding=1, stride=1
    You should 1) load the values from input and kernel, 2) do computation, 3) store the result
    
    Each program instance computes one output element O[k, x, y].
    Grid: (C_OUT, H, W)
    """
    # TODO: Complete the triton kernel that does convolution
    
    # Which output element are we computing??
    k = tl.program_id(0)         # output channel
    out_x = tl.program_id(1)     # output row
    out_y = tl.program_id(2)     # output col

    # Declare accumulator
    acc = tl.zeros([], dtype=tl.float32)

    # Iterate... a lot, over input channels and filter taps
    for c in range(C_IN):
        for fh in range(FH):
            for fw in range(FW):
                # Inputs
                in_x = out_x + fh - PAD
                in_y = out_y + fw - PAD

                # Check bounds
                in_bounds = (in_x >= 0) & (in_x < H) & (in_y >= 0) & (in_y < W)

                # Load input value (0 if out of bounds = zero padding)
                # input layout: (1, C_IN, H, W)
                # offset = c*H*W + in_x*W + in_y
                input_offset = c * H * W + in_x * W + in_y
                input_val = tl.load(input_ptr + input_offset, mask=in_bounds, other=0.0)

                # Load filter value
                # kernel layout: (C_OUT, C_IN, FH, FW)
                # offset = k*C_IN*FH*FW + c*FH*FW + fh*FW + fw
                kernel_offset = k * C_IN * FH * FW + c * FH * FW + fh * FW + fw
                kernel_val = tl.load(kernel_ptr + kernel_offset)

                acc += input_val * kernel_val

    # Store result
    # output layout: (1, C_OUT, H, W)
    # offset = k*H*W + out_x*W + out_y
    output_offset = k * H * W + out_x * W + out_y
    tl.store(output_ptr + output_offset, acc)


def my_conv2d(input, kernel):
    """
    This function is a wrapper function that preprocess the inputs and call the kernel
    input: torch.tensor (1, C_IN, H, W)
    kernel: torch.tensor (C_OUT, C_IN, FH, FW)
    """
    # TODO: Initializing some variables
    _, C_IN, H, W = input.shape
    C_OUT, _, FH, FW = kernel.shape
    PAD = 1

    # TODO: Calculate output dimension & Allocate output tensor
    OUT_H = H
    OUT_W = W
    output = torch.empty(1, C_OUT, OUT_H, OUT_W, device=input.device, dtype=input.dtype)

    # TODO: Define grid
    # One program per output element
    grid = (C_OUT, OUT_H, OUT_W)

    # TODO: Call the triton kernel (my_triton_kernel) and measure execution time
    # Warmup
    my_triton_kernel[grid](
        input, kernel, output,
        C_IN, H, W, C_OUT, FH, FW, PAD,
    )
    torch.cuda.synchronize()

    # Time it!
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)

    start.record()
    my_triton_kernel[grid](
        input, kernel, output,
        C_IN, H, W, C_OUT, FH, FW, PAD,
    )
    end.record()
    torch.cuda.synchronize()

    execution_time = start.elapsed_time(end)  # in milliseconds

    # TODO: Return output (output should include execution time)
    return output, f"{execution_time:.3f} ms"

In [ ]:
# Testing
# Comparing the result from my_conv2d and Conv from torch
my_output, execution_time = my_conv2d(tensor_I, tensor_F)
torch.testing.assert_close(golden_out, my_output)   # Assert statement should be passed

# Printing the execution time *fingers crossed*
print(f"Execution Time for triton kernel: {execution_time}")